# grind -d dir -m pat -r recursively
Implement Linux grind (combination of grep and find) with options -m for matching the name with the given pattern, -d the path and -r if recuresively done. 

In [ ]:
import argparse
import fnmatch
import os
import sys
from pathlib import Path


def parse_args():
    parser = argparse.ArgumentParser(
        description="Search for matching files/directories under a directory."
    )

    parser.add_argument(
        "-d",
        "--directory",
        required=True,
        help="Root directory to search from",
    )

    parser.add_argument(
        "-m",
        "--match",
        required=True,
        help='Filename pattern to match, for example "*.log" or "api*"',
    )

    parser.add_argument(
        "-r",
        "--recursive",
        action="store_true",
        help="Search recursively through subdirectories",
    )

    return parser.parse_args()


def format_output(path: Path, root: Path) -> str:
    """
    Return path relative to root.
    Add '/' at the end if this path is a directory.
    """

    relative_path = path.relative_to(root)

    output = str(relative_path)

    if path.is_dir():
        output += "/"

    return output


def search_non_recursive(root: Path, pattern: str):
    """
    Search only the direct children of root.
    Do not descend into subdirectories.
    """

    try:
        for entry in root.iterdir():
            if fnmatch.fnmatch(entry.name, pattern):
                print(format_output(entry, root))

    except OSError as e:
        print(f"Error reading directory {root}: {e}", file=sys.stderr)


def search_recursive(root: Path, pattern: str):
    """
    Recursively search all descendants of root.
    Do not print root itself.
    """

    try:
        for current_dir, dirnames, filenames in os.walk(root):
            current_path = Path(current_dir)

            # Check directories
            for dirname in dirnames:
                dir_path = current_path / dirname

                if fnmatch.fnmatch(dirname, pattern):
                    print(format_output(dir_path, root))

            # Check files
            for filename in filenames:
                file_path = current_path / filename

                if fnmatch.fnmatch(filename, pattern):
                    print(format_output(file_path, root))

    except OSError as e:
        print(f"Error walking directory {root}: {e}", file=sys.stderr)


def main():
    args = parse_args()

    root = Path(args.directory)

    if not root.exists():
        print(f"Error: directory does not exist: {root}", file=sys.stderr)
        sys.exit(1)

    if not root.is_dir():
        print(f"Error: path is not a directory: {root}", file=sys.stderr)
        sys.exit(1)

    if args.recursive:
        search_recursive(root, args.match)
    else:
        search_non_recursive(root, args.match)


if __name__ == "__main__":
    main()

Using os.listdir instead of walk and os.path.relpath

In [ ]:
import argparse
import fnmatch
import os
import sys


def parse_args():
    parser = argparse.ArgumentParser(
        description="Simplified find-like CLI."
    )

    parser.add_argument(
        "-d",
        "--directory",
        required=True,
        help="Root directory to search from",
    )

    parser.add_argument(
        "-m",
        "--match",
        default="*",
        help='Optional glob pattern to match names, for example "*.log". Defaults to "*"',
    )

    parser.add_argument(
        "-r",
        "--recursive",
        action="store_true",
        help="Search recursively",
    )

    return parser.parse_args()


def print_match(full_path, root):
    """
    Print path relative to the original root.
    Add '/' when the path is a directory.
    """

    relative_path = os.path.relpath(full_path, root)

    if os.path.isdir(full_path):
        relative_path += "/"

    print(relative_path)


def should_match(name, pattern):
    """
    Match the filename or directory name against the pattern.
    Since pattern defaults to '*', this returns True for everything
    when -m is not provided.
    """

    return fnmatch.fnmatch(name, pattern)


def search_non_recursive(root, pattern):
    """
    Search only direct children of root.
    """

    try:
        for name in os.listdir(root):
            full_path = os.path.join(root, name)

            if should_match(name, pattern):
                print_match(full_path, root)

    except OSError as e:
        print(f"Error reading directory {root}: {e}", file=sys.stderr)


def search_recursive(current_dir, root, pattern):
    """
    Recursively search using os.listdir().
    """

    try:
        for name in os.listdir(current_dir):
            full_path = os.path.join(current_dir, name)

            if should_match(name, pattern):
                print_match(full_path, root)

            if os.path.isdir(full_path):
                search_recursive(full_path, root, pattern)

    except OSError as e:
        print(f"Error reading directory {current_dir}: {e}", file=sys.stderr)


def main():
    args = parse_args()

    root = args.directory
    pattern = args.match

    if not os.path.exists(root):
        print(f"Error: path does not exist: {root}", file=sys.stderr)
        sys.exit(1)

    if not os.path.isdir(root):
        print(f"Error: path is not a directory: {root}", file=sys.stderr)
        sys.exit(1)

    if args.recursive:
        search_recursive(root, root, pattern)
    else:
        search_non_recursive(root, pattern)


if __name__ == "__main__":
    main()